## 4. Задание: пишем игру целиком

Теперь у нашего парсера команд есть все, что нужно. Соберем из него
настоящую игру. Игровой мир уже описан — за него отвечают три функции,
написанные тоже через `match`:

- `get_next_room(room, direction)` — куда ведет выход из комнаты (или `None`,
  если выхода нет). Субъектом `match` здесь служит **кортеж**
  `(комната, направление)`, и образцы сопоставляют его целиком;
- `describe_room(room)` — текстовое описание комнаты;
- `list_items(room, inventory)` — предметы, которые лежат в комнате и еще не
  подобраны игроком.

Изучите их, а потом запустите ячейку и поэкспериментируйте.

### Что нужно сделать

Игрок находится в зале (`hall`) с пустым инвентарем. Цель игры — найти в
библиотеке ключ и открыть им калитку в саду. Реализуйте цикл обработки
команд с помощью `match`-`case`, чтобы игра отвечала на следующие команды
(тексты сообщений должны быть именно такими):

| Команда | Что делает |
|---|---|
| `quit` или `exit` | выводит `Goodbye!` и завершает игру |
| `look` | выводит описание текущей комнаты, а затем `Items here: [...]` — доступные в ней предметы |
| `inventory` | выводит `You have: [...]` — список подобранных предметов |
| `go <направление>` | если выход есть — переходит в другую комнату и выводит ее описание, иначе выводит `You can't go that way` |
| `get <предмет>` или `take <предмет>` | если предмет доступен в комнате — кладет его в инвентарь и выводит `You picked up: <предмет>`, иначе выводит `There is no <предмет> here` |
| `open gate` | если игрок в саду (`garden`) и у него есть `key` — выводит `You opened the gate with the key. You win!` и завершает игру, иначе выводит `The gate is locked or too far away` |
| пустая строка | выводит `Say something!` |
| любая другая | выводит `Unknown command: <первое слово команды>` |

**Подсказки.**

- Заготовка цикла ниже уже содержит обработку `quit` / `exit` — продолжайте
  по аналогии. Нужные вам переменные `current_room`, `inventory` и
  `available_items` (предметы, доступные в текущей комнате) уже есть.
- Синонимы `quit` / `exit` и `get` / `take` удобно разобрать **or-образцом**.
- Команда `go` — хороший случай для guard вместе с моржовым оператором: в
  условии нужно **и проверить**, что выход существует, **и получить**
  название комнаты, куда он ведет. А в теле `case` использовать готовое
  значение:

  ```python
  case ["go", direction] if (next_room := get_next_room(current_room, direction)):
      ...
  ```
- Для `get` / `take` и `open gate` понадобятся guard'ы: сначала более
  конкретный `case` с условием, ниже — общий для случая, когда условие
  не выполнилось.
- Пустую команду ловит образец `[]`, а любую непустую — образец
  `[first_word, *_]` (первое слово запоминаем, остальные отбрасываем). Вместе
  они покрывают все возможные списки слов, поэтому `case _` не нужен.
- Помните: **порядок `case` важен** — конкретные образцы выше общих.

Для запуска ячейки в VS Code поле для ввода появится в верхней части окна.

In [ ]:
def get_next_room(room: str, direction: str) -> str | None:
    match (room, direction):
        case ("hall", "north"):
            return "library"

        case ("hall", "east"):
            return "garden"

        case ("library", "south") | ("garden", "west"):
            return "hall"

        case _:
            return None


def describe_room(room: str) -> str:
    match room:
        case "hall":
            return "You are in the hall. Exits: north, east."

        case "library":
            return "You are in the library. Exits: south."

        case "garden":
            return "You are in the garden. A locked gate is here. Exits: west."

        case _:
            return "You are nowhere."


def list_items(room: str, inventory: list[str]) -> list[str]:
    match room:
        case "hall":
            all_items = ["lamp"]

        case "library":
            all_items = ["book", "key"]

        case _:
            all_items = []

    return [item for item in all_items if item not in inventory]


print(get_next_room("hall", "north"))  # library
print(get_next_room("hall", "up"))     # None
print(describe_room("garden"))
print(list_items("library", ["book"]))  # ['key']

In [ ]:
current_room = "hall"
inventory: list[str] = []

print(describe_room(current_room))

while True:
    command = input("> ")
    available_items = list_items(current_room, inventory)

    match command.split():
        case ["quit"] | ["exit"]:
            print("Goodbye!")
            break

        case ["look"]:
            print(describe_room(current_room))
            print(f"Items here: {available_items}")

        case ["inventory"]:
            print(f"You have: {inventory}")

        case ["go", direction] if (next_room := get_next_room(current_room, direction)):
            current_room = next_room
            print(describe_room(current_room))
        
        case ["go", _]:
            print("You can't go that way")

        case ["get", item] | ["take", item] if item in available_items:
            print(f"You picked up: {item}")
            inventory.append(item)

        case ["get", item] | ["take", item]:
            print(f"There is no {item} here")

        case ["open", "gate"] if "key" in inventory and current_room == "garden":
            print("You opened the gate with the key. You win!")
            break

        case ["open", "gate"]:
            print("The gate is locked or too far away")

        case []:
            print("Say something!")

        case [first_word, *_]:
            print(f"Unknown command: {first_word}")